# 模板

FastAPI 支持多种模板引擎。

Flask 等工具使用的 Jinja2 是最用的模板引擎。

在 Starlette 的支持下，FastAPI 应用可以直接使用工具轻易地配置 Jinja2。

## 安装依赖项
### 安装 jinja2：

`pip install jinja2`

## 使用 Jinja2Templates¶
- 导入 Jinja2Templates
- 创建可复用的 templates 对象
- 在返回模板的路径操作中声明 Request 参数
- 使用 templates 渲染并返回 TemplateResponse， 传递模板的名称、request对象以及一个包含多个键值对（用于Jinja2模板）的"context"字典，

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates

app = FastAPI()

app.mount("/static", StaticFiles(directory="static"), name="static")


templates = Jinja2Templates(directory="templates")


@app.get("/items/{id}", response_class=HTMLResponse)
async def read_item(request: Request, id: str):
    return templates.TemplateResponse(
        request=request, name="item.html", context={"id": id}
    )

> 在FastAPI 0.108.0，Starlette 0.29.0之前，name是第一个参数。 并且，在此之前，request对象是作为context的一部分以键值对的形式传递的。

> 通过声明 response_class=HTMLResponse，API 文档就能识别响应的对象是 HTML。

> 您还可以使用 from starlette.templating import Jinja2Templates。

> FastAPI 的 fastapi.templating 只是为开发者提供的快捷方式。实际上，绝大多数可用响应都直接继承自 Starlette。 Request 与 StaticFiles 也一样。

## 编写模板¶
编写模板 templates/item.html，代码如下：

```HTML
<html>
<head>
    <title>Item Details</title>
    <link href="{{ url_for('static', path='/styles.css') }}" rel="stylesheet">
</head>
<body>
    <h1><a href="{{ url_for('read_item', id=id) }}">Item ID: {{ id }}</a></h1>
</body>
</html>
```


### 模板上下文¶
在包含如下语句的html中:

`Item ID: {{ id }}`

...这将显示你从"context"字典传递的 id:

`{"id": id}`

例如。当ID为 42时, 会渲染成:

`Item ID: 42`

### 模板 url_for 参数¶
你还可以在模板内使用 `url_for()`，其参数与路径操作函数的参数相同.

所以，该部分:
`<a href="{{ url_for('read_item', id=id) }}">`

...将生成一个与处理路径操作函数 `read_item(id=id)`的URL相同的链接

例如。当ID为 42时, 会渲染成:

`<a href="/items/42">`


### 模板与静态文件¶
你还可以在模板内部将 url_for()用于静态文件，例如你挂载的 name="static"的 StaticFiles。

```html
<html>
<head>
    <title>Item Details</title>
    <link href="{{ url_for('static', path='/styles.css') }}" rel="stylesheet">
</head>
<body>
    <h1><a href="{{ url_for('read_item', id=id) }}">Item ID: {{ id }}</a></h1>
</body>
</html>

```

本例中，它将链接到 static/styles.css中的CSS文件：

```css

h1 {
    color: green;
}
```


因为使用了 StaticFiles， FastAPI 应用会自动提供位于 URL /static/styles.css的 CSS 文件。

### 更多说明¶
包括测试模板等更多详情，请参阅 Starlette 官方文档 - 模板。